## 课程一：MeloTTS 的安装（15%）

实验目标：成功安装 MeloTTS 并运行示例代码

MeloTTS is a **high-quality multi-lingual** text-to-speech library by [MIT](https://www.mit.edu/) and [MyShell.ai](https://myshell.ai/). Support English, Spanish, French, Chinese, Japanese and Korean.


### 1、环境配置

从 Github 上下载代码，按照 install.md 的说明进行安装:

```bash
# 从远程仓库拉取代码
git clone https://github.com/CJY1018/MeloTTS-Homework.git
cd MeloTTS-Homework

# 创建 conda 虚拟环境
conda create -n melotts python=3.9 -y
conda activate melotts


# 安装依赖
pip install -e .

# 下载 UniDic 词典到本地，避免因网络问题导致 unidic 安装失败
modelscope download --model CJY1018/dicdir --local_dir ./dicdir
```

### 2、使用 Python API 的方式进行推理

使用 HF-Mirror 代理，若后续碰到网络问题，都可以添加下面的代码后重启 ipykernel 内核再试试：

In [1]:
import os

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

In [2]:
from melo.api import TTS


# Speed is adjustable
speed = 1.0
# 可使用 CPU 或 GPU 进行推理，若使用 GPU，确保已正确安装 CUDA 和 PyTorch 的 GPU 版本，并将 device 设置为 'cuda:0' 或相应的 GPU 设备编号
device = 'cuda:0' # or cuda:0

text = "我最近在学习machine learning，希望能够在未来的artificial intelligence领域有所建树。"
model = TTS(language='ZH', device=device)
speaker_ids = model.hps.data.spk2id

output_path = 'zh.wav'
model.tts_to_file(text, speaker_ids['ZH'], output_path, speed=speed) # melo/api.py

C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.23). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth

C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


 > Text split to sentences.
我最近在学习machine learning,
希望能够在未来的artificial intelligence领域有所建树.
 > ===========================


  0%|          | 0/2 [00:00<?, ?it/s]

...

Prefix dict has been built successfully.


Some weights of the model checkpoint at bert-base-multilingual-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


 50%|█████     | 1/2 [00:15<00:15, 15.69s/it]

100%|██████████| 2/2 [00:16<00:00,  6.65s/it]

100%|██████████| 2/2 [00:16<00:00,  8.01s/it]

In [3]:
# 播放生成的音频
from IPython.display import Audio

Audio(output_path)

> 【问题一】MeloTTS 的 Python API (`melo/api.py`) 中，`tts_to_file` 方法有以下参数：`text`, `speaker_id`, `output_path`, `sdp_ratio`, `noise_scale`, `noise_scale_w`, `speed`, `pbar`, `format`, `position`, `quiet`。请解释每个参数的含义及作用，可结合 melo/api.py 源码和实际修改参数后的合成效果进行说明。

**`tts_to_file` 各参数含义及作用：**

1. **`text`**：待合成的输入文本字符串。在方法内部会通过 `split_sentences_into_pieces` 按标点切分为多个子句，逐句合成后拼接。

2. **`speaker_id`**：说话人 ID，用于多说话人模型中选择不同的音色。通过 `model.hps.data.spk2id` 字典获取。在模型内部通过 `self.emb_g(sid)` 提取对应的说话人嵌入向量。

3. **`output_path`**：输出音频文件的保存路径。若为 `None`，则直接返回音频的 numpy 数组；否则使用 `soundfile.write` 写入文件。

4. **`sdp_ratio`**（默认 0.2）：控制**随机时长预测器（Stochastic Duration Predictor, SDP）**与**确定性时长预测器（Duration Predictor, DP）**的混合比例。`sdp_ratio=0` 时完全使用 DP（输出确定性），`sdp_ratio=1` 时完全使用 SDP（输出随机性）。计算公式：`logw = sdp(...) * sdp_ratio + dp(...) * (1 - sdp_ratio)`。增大此值会使语音韵律更自然多变，但也可能引入不稳定性。

5. **`noise_scale`**（默认 0.6）：控制从先验分布中采样隐变量 `z_p` 时的噪声幅度，即 `z_p = m_p + randn * exp(logs_p) * noise_scale`。值越大，生成语音的多样性和随机性越高；值越小，语音越稳定但可能略显单调。

6. **`noise_scale_w`**（默认 0.8）：控制 SDP 模块中采样时长时的噪声幅度。值越大，预测的音素时长变化越大，韵律越丰富；值越小，时长越稳定。

7. **`speed`**（默认 1.0）：语速倍率。内部通过 `length_scale = 1.0 / speed` 传入模型，对每个音素的时长进行缩放。`speed=0.5` 时语速减半（时长翻倍），`speed=2.0` 时语速加倍（时长减半）。

8. **`pbar`**：自定义的进度条函数，用于替换默认的 `tqdm` 进度条。若提供，则调用 `pbar(texts)` 来迭代子句列表。

9. **`format`**：输出音频的文件格式（如 `'wav'`, `'mp3'` 等）。若为 `None`，则由 `soundfile.write` 根据文件扩展名自动判断。

10. **`position`**：在多进程/多线程场景下，指定 `tqdm` 进度条的显示位置（行号），避免多条进度条互相覆盖。

11. **`quiet`**（默认 `False`）：静默模式。为 `True` 时不显示进度条，也不打印句子切分信息，适用于批量推理场景。